<a href="https://colab.research.google.com/github/FishyFoshy/COMP3608-Project/blob/main/Dataset_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cleaning of Dataset 1

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load datasets
df1 = pd.read_csv('Dataset 1.csv')

# Removes rows that don't have HIGH confidence in their sales estimates
df1 = df1[(df1['saleEstimate_confidenceLevel'] == 'HIGH')]

# Drops all unnecessary columns
df1 = df1.drop(columns=['fullAddress', 'postcode', 'rentEstimate_lowerPrice', 'rentEstimate_currentPrice', 'rentEstimate_upperPrice', 
                        'saleEstimate_lowerPrice', 'saleEstimate_upperPrice', 'saleEstimate_ingestedAt', 'saleEstimate_valueChange.numericChange', 
                        'saleEstimate_valueChange.percentageChange', 'saleEstimate_valueChange.saleDate', 'history_date', 'history_price', 'history_percentageChange', 
                        'history_numericChange', 'saleEstimate_confidenceLevel'])

# Drops all with null values
df1.dropna(inplace=True)
df1['saleEstimate_currentPrice'] = np.log(df1['saleEstimate_currentPrice'])

# Separates non numerical columns from numerical ones
non_numerical_columns = ['saleEstimate_currentPrice', 'tenure', 'propertyType', 'currentEnergyRating']
numerical_features = [col for col in df1.columns if col not in non_numerical_columns]

# Scale numerical features
scaler = StandardScaler()
df1[numerical_features] = scaler.fit_transform(df1[numerical_features])

# One-Hot Encodes all categorical data
categorical_cols_to_encode = df1.select_dtypes(include='object').columns
df1 = pd.get_dummies(df1, columns=categorical_cols_to_encode, drop_first=True)

# Identifies target column from the features
target_column = 'saleEstimate_currentPrice'
features = [col for col in df1.columns if col != target_column]

x = df1[features].copy()
y = df1[target_column].copy()

# Splits the data 80/20 for training and testing the model respectfully
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("=" * 60)
print("DATASET 1: Dataset 1.csv")
print("=" * 60)
print("\nFirst 10 rows:")
display(df1.head(10))
print("\nData types:")
print(df1.dtypes)
print("\nMissing values:")
print(df1.isnull().sum())
print("\nTarget variable (saleEstimate_currentPrice) statistics:")
print(df1['saleEstimate_currentPrice'].describe())

In [ ]:
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import numpy as np
# Inverse the log transformation to its original scale

def rmse_original_scale(y_true, y_pred):
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate RMSE on the original scale
    return np.sqrt(mean_squared_error(y_true_original, y_pred_original))

def mae_original_scale(y_true, y_pred):
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate MAE on the original scale
    return mean_absolute_error(y_true_original, y_pred_original)

def r2_original_scale(y_true, y_pred):
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate R2 on the original scale
    return r2_score(y_true_original, y_pred_original)

# Create scorers that can be used with cross_val_score
neg_rmse_original_scorer = make_scorer(rmse_original_scale, greater_is_better=False)
neg_mae_original_scorer = make_scorer(mae_original_scale, greater_is_better=False)
r2_original_scorer = make_scorer(r2_original_scale, greater_is_better=True)

### Linear Regression for Dataset 1

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

# Train baseline Linear Regression
lr = LinearRegression()
lr.fit(x_train, y_train)
y_pred_lr = lr.predict(x_test)

# Inverse log transform to get actual prices
y_test_actual = np.exp(y_test)
y_pred_lr_actual = np.exp(y_pred_lr)

# Evaluate on original scale
rmse_lr = np.sqrt(mean_squared_error(y_test_actual, y_pred_lr_actual))
mae_lr = mean_absolute_error(y_test_actual, y_pred_lr_actual)
r2_lr = r2_score(y_test_actual, y_pred_lr_actual)

print("=" * 50)
print("BASELINE: Linear Regression on Dataset 1.csv")
print("=" * 50)
print(f"RMSE:\t${rmse_lr:,.2f}")
print(f"MAE:\t${mae_lr:,.2f}")
print(f"R²:\t{r2_lr:.4f}")
print(f"\nCross-validation RMSE:\t${-cross_val_score(lr, x, y, cv=5, scoring=neg_rmse_original_scorer).mean():,.2f}")
print(f"Cross-validation MAE:\t${-cross_val_score(lr, x, y, cv=5, scoring=neg_mae_original_scorer).mean():,.2f}")
print(f"Cross-validation R²:\t{cross_val_score(lr, x, y, cv=5, scoring=r2_original_scorer).mean():.4f}")

# Predicted vs Actual scatter
plt.figure(figsize=(8, 6))
plt.scatter(y_test_actual, y_pred_lr_actual, alpha=0.4, s=15)
plt.plot([0, y_test_actual.max()], [0, y_test_actual.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Baseline Linear Regression: Predicted vs Actual House Price')
plt.legend()
plt.tight_layout()
plt.show()

# Feature Importance Plot

In [ ]:
import seaborn as sns

# Fetching Feature Names and Coefficients
feature_names = x_train.columns
coefficients = lr.coef_

# Wrapping them together in a Pandas DataFrame for Easy Comparison and Viewing
feature_importance_lr = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Using the Absolute Function before Sorting the Coefficients in Descending Order
feature_importance_lr['Abs_Coefficient'] = np.abs(feature_importance_lr['Coefficient'])
top_10_features_lr = feature_importance_lr.sort_values(by='Abs_Coefficient', ascending=False).head(10)

# Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_10_features_lr,
    x='Coefficient',
    y='Feature',
)
plt.title('Top 10 Features Driving Property Prices (Linear Regression)', fontsize=15)
plt.xlabel('Coefficient Value (Impact on Price)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features_lr[['Feature', 'Coefficient']])

# Analysis

Linear Regression model works well with the low to mid range properties, but starts to grossly underestimate when the prices go above about £3,000,000. This is validated by the scatter plot whose predictions were concentrated around the range of £3-4M whereas the actual prices were up to 9M which means that the model fails to reflect the variance in the higher price range.

The most significant characteristic is easily floorAreaSqM, whose coefficient is around +£530,000, i.e. when the floor area increases by one standard deviation, the price at which the house is predicted to sell will increase by more than half a million pounds. This is logical, larger properties are much more expensive. A combination of property types and energy ratings are the next most significant characteristics. The large negative coefficients of the following property types include: propertytype_Semi-Detached Property, propertytype_End Terrace Property, and propertytype_Purpose Built Flat (which all have a large negative coefficient of around -130,000 to -170,000), so these property types are linked to lower sale prices than the base category (likely detached houses). This is logical because free standing (detached) properties are generally at the very top of the UK housing market.

Interestingly, all the energy ratings (E, G, F, D, C) are all in the top 10 with positive coefficients between £150,000 and 200,000. These positive coefficients appear counterintuitive since the highest energy rating (A or B) is probably the default since one-hot encoding with drop first=True is used, so the lower the energy rating, it should be the more expensive it is. This is, however, probably explained by the fact that older, larger, and more prestigious properties (e.g. period homes, Victorian terraces) have lower energy ratings with a high price tag. This correlation is one that the model is picking up, and not a real causal effect of poor energy efficiency raising the value.

The coefficient of tenure_Freehold is negative (approximately -100,000) which is not expected as freehold properties tend to be more attractive. This once more can be attributed to an outside factor: leasehold property included in this data can be biased towards more expensive London flats, which inflates the baseline leasehold price.

The major weakness of the Linear Regression model is that it does not show good performance at the high end where the RMSE is much greater than the MAE - a feature of large outlier errors. The model is inherently restricted because it is linear: it does not even allow the non-linear relations between features (e.g. a large floor area in a prime London longitude attracts a much higher price). These interactions should be able to be captured by a non-linear model like the Random Forest and enhance predictions at the upper end of the price distribution.

# Random Forest Regression for Dataset 1

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

rf = RandomForestRegressor(n_estimators=265, n_jobs=5, max_features=0.95, max_depth=50, random_state=42)
rf.fit(x_train, y_train)
y_pred_rf = rf.predict(x_test)

# Inverse log transform to get actual prices
y_pred_rf_actual = np.exp(y_pred_rf)

# Evaluate on original scale
rmse_rf = np.sqrt(mean_squared_error(y_test_actual, y_pred_rf_actual))
mae_rf = mean_absolute_error(y_test_actual, y_pred_rf_actual)
r2_rf = r2_score(y_test_actual, y_pred_rf_actual)

print("=" * 50)
print("Random Forest Regressor on Dataset 1.csv")
print("=" * 50)
print(f"RMSE:\t${rmse_rf:,.2f}")
print(f"MAE:\t${mae_rf:,.2f}")
print(f"R²:\t{r2_rf:.4f}")

print(f"\nCross-validation RMSE:\t${-cross_val_score(rf, x, y, cv=5, scoring=neg_rmse_original_scorer).mean():,.2f}")
print(f"Cross-validation MAE:\t${-cross_val_score(rf, x, y, cv=5, scoring=neg_mae_original_scorer).mean():,.2f}")
print(f"Cross-validation R²:\t{cross_val_score(rf, x, y, cv=5, scoring=r2_original_scorer).mean():.4f}")

plt.figure(figsize=(8, 6))
plt.scatter(y_test_actual, y_pred_rf_actual, alpha=0.4, s=15)
plt.plot([0, y_test_actual.max()], [0, y_test_actual.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Random Forest Regressor: Predicted vs Actual House Price')
plt.legend()
plt.tight_layout()
plt.show()

# Feature Importance Plot

In [ ]:
# Fetching Feature Names and Importances from Random Forest
feature_names = x_train.columns
importances = rf.feature_importances_

# Wrapping them together in a Pandas DataFrame for Easy Comparison and Viewing
feature_importance_rf = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sorting Feature Importances in Descending Order
top_10_features_rf = feature_importance_rf.sort_values(by='Importance', ascending=False).head(10)

# Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_10_features_rf,
    x='Importance',
    y='Feature',
)
plt.title('Top 10 Features Driving Property Prices (Random Forest)', fontsize=15)
plt.xlabel('Feature Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features_rf[['Feature', 'Importance']])

# Analysis

The story of what drives property prices as explained by the Random Forest model is quite different than the Linear Regression. The continuous variables that dominate the majority of this feature importance chart are: floorAreaSqM (~0.65), longitude (~0.16) and latitude (~0.13). Combining the three features, the overall predictive power of the model is computed to be around 94 percent, whereas all other features have only small significant value.

The most significant feature, which is floorareaSqM, is not only consistent with the results of the Linear Regression but also makes sense in the real world, being that property size is the one of, if not the most important characteristic to predict price. But the most interesting thing about the Random Forest model is the prominence of longitude and latitude. These geographic coordinates were hardly visible in the most popular features of the Linear Regression as the impact of location on price is very non-linear. For instance, a property in central London (specific longitude/latitude ranges) can be worth 5-10x the price of a similar property elsewhere. The Random Forest can capture these non-linear spatial patterns through its tree-based splits, effectively learning "if longitude is between X and Y and latitude is between A and B, then the property is in a premium area."

The other attributes such as "bedrooms, bathrooms, tenure Freehold, living Rooms and other types of property have lower contribution of less than 1-2 percent each. This does not imply they are irrelevant, but that once the floor area and location is known, they contribute a marginal amount to the predictive signal. This is in part due to the fact that much of the information that will be given by bedrooms and bathrooms is already encoded in the floor area (larger homes naturally have more rooms).

The scatter plot proves the superiority of the Random Forest: the predictions follow the diagonal line far better through the whole price range, even the higher bracket where Linear Regression failed. The RMSE is much lower than the Linear Regression and the difference between the RMSE and MAE is much less, meaning that the model no longer suffers the extreme outlier errors that affected the linear model on the high end.

The general lesson learned is that in this case, is that property prices depend mostly on size and location, and the non-linear geographic pricing patterns that Linear Regression can not identify can be captured by the Random Forest. The fact that the model only uses three features to make 94% of its predictions also indicates that a finer location-based features (e.g. proximity to transport, school ratings, neighbourhood crime rates) may be useful in this dataset to give it additional predictive power over raw coordinates.